# **MDANCE quick start: clustering with NANI**
**Molecular Dynamics Analysis with N-ary Clustering Ensembles (MDANCE)** is a flexible n-ary clustering package that provides a set of tools for clustering Molecular Dynamics trajectories. The package is designed to be modular and extensible, allowing for the addition of new clustering algorithms and similarity metrics. You can find more information here: https://mdance.readthedocs.io/

This notebook walks through one method, NANI, from a trajectory to clustered frames. The other methods in MDANCE are listed at the end of the notebook.

Adapted from a notebook by [Pablo Ricardo Arantes](https://pablo-arantes.github.io/)

# **Overview**
This clustering notebook is meant for Molecular Dynamics datasets. We will use the k-Means N-Ary Natural Initiation (NANI) algorithm to select initial centroids on k-Means clustering. NANI is an extension of the k-Means++ algorithm. NANI stratifies the data into high-density regions and performs diversity selection on top of them to select the initial centroids. This is a deterministic algorithm that will always select the same initial centroids for the same dataset and improve on k-means++ by reducing the number of iterations required to converge and improve the clustering quality.

For more information on the NANI algorithm, please refer to the NANI paper: https://doi.org/10.1021/acs.jctc.4c00308.

In [ ]:
#@title **Install MDANCE**
#@markdown Installs MDANCE and its dependencies from PyPI.
%pip install -q mdance

In [ ]:
# @title ### **Preprocessing of Molecular Dynamics Data**
#@markdown This cell reads a trajectory and saves the coordinates of the selected atoms as a numpy array. Make sure the trajectory is already aligned and/or centered if needed.

import numpy as np

from mdance import data
from mdance.inputs.preprocess import gen_traj_numpy

#@markdown **use_sample_data** runs the notebook on the sample trajectory included with MDANCE. To use your own data, uncheck it and upload your topology and trajectory files to Colab.
use_sample_data = True #@param {type:"boolean"}

#@markdown Enter the path for the trajectory file, without the extension (only used if **use_sample_data** is unchecked):
trajectory_file = 'my_trajectory' #@param {type:"string"}
input_format_trajectory = "dcd" #@param ["pdb", "dcd", "nc", "xtc", "gro", "trr"]

#@markdown Enter the path for the topology file, without the extension (only used if **use_sample_data** is unchecked):
topology_file = 'my_topology' #@param {type:"string"}
input_format_topology = "pdb" #@param ["pdb", "gro", "top", "tpr", "prmtop", "parm7", "psf"]

#@markdown The atom selection used for clustering that must be compatible with the [MDAnalysis Atom Selections Language](https://userguide.mdanalysis.org/stable/selections.html). The default selects the backbone of residues 3 to 12 in the sample data.
selection = 'resid 3 to 12 and name N CA C O H' #@param {type:"string"}
output_file_name = 'tau' #@param {type:"string"}

if use_sample_data:
    input_traj = str(data.traj)
    input_top = str(data.top)
else:
    input_traj = f'{trajectory_file}.{input_format_trajectory}'
    input_top = f'{topology_file}.{input_format_topology}'

traj_numpy = gen_traj_numpy(input_top, input_traj, selection)
N_atoms = traj_numpy.shape[1] // 3
output_name = output_file_name + '.npy'
np.save(output_name, traj_numpy)

In [ ]:
# @title ### **NANI Screening**
#@markdown This cell runs NANI for a range of cluster numbers and calculates cluster quality metrics. For the best result, we recommend screening a wide range of cluster numbers.
import os

import numpy as np

from mdance.cluster.nani import KmeansNANI, compute_scores
from mdance.tools.bts import extended_comparison

#@markdown **sieve** takes every sieve-th frame from the trajectory for analysis.
sieve = 1 #@param {type:"integer"}

#@markdown **start_n_clusters** is the starting number for screening. It must be at least 2.
start_n_clusters = 2 #@param {type:"integer"}

#@markdown **end_n_clusters** is the ending number for screening.
end_n_clusters = 30 #@param {type:"integer"}

# NANI parameters
output_dir = f'outputs_{output_file_name}'
init_type = 'comp_sim'
metric = 'MSD'
percentage = 10

os.makedirs(output_dir, exist_ok=True)
traj_numpy = np.load(output_name)[::sieve]

# The initiators are selected once; each number of clusters k uses the first k of them
mod = KmeansNANI(data=traj_numpy, n_clusters=end_n_clusters, metric=metric,
                 N_atoms=N_atoms, init_type=init_type, percentage=percentage)
initiators = mod.initiate_kmeans()

all_scores = []
for i in range(start_n_clusters, end_n_clusters + 1):
    # Run k-means clustering
    mod = KmeansNANI(data=traj_numpy, n_clusters=i, metric=metric,
                     N_atoms=N_atoms, init_type=init_type, percentage=percentage)
    labels, centers, n_iter = mod.kmeans_clustering(initiators[:i])

    # Compute scores
    ch_score, db_score = compute_scores(data=traj_numpy, labels=labels)

    # Calculate MSD for each cluster
    total = 0
    for j in range(i):
        cluster = traj_numpy[labels == j]
        total += extended_comparison(cluster, data_type='full', metric=metric, N_atoms=N_atoms)
    all_scores.append((i, n_iter, ch_score, db_score, total / i))

all_scores = np.array(all_scores)
header = f'init_type: {init_type}, percentage: {percentage}, metric: {metric}, sieve: {sieve}\n'
header += 'Number of clusters, Number of iterations, Calinski-Harabasz score, Davies-Bouldin score, Average MSD'
np.savetxt(f'{output_dir}/{output_file_name}_summary.csv', all_scores,
           delimiter=',', header=header, fmt='%s')

In [ ]:
# @title ### **Cluster Analysis**
#@markdown The output from last cell (NANI screening) will be used to plot the optimal number of clusters.

#@markdown The function below will plot the Davies-Bouldin index and the optimal number of clusters.
#@markdown The optimal number of clusters is determined by the minimum Davies-Bouldin index or the minimum of the second derivative of the Davies-Bouldin index.


import matplotlib.pyplot as plt
import numpy as np

scores_csvs = [f'{output_dir}/{output_file_name}_summary.csv']

def plot_scores(scores_csv):
    """Plot the Davies-Bouldin index and the optimal number of clusters.

    Parameters
    ----------
    scores_csv : str
        The path to the csv file that contains the screening

    Returns
    -------
    file
        A png file that contains the plot of the Davies-Bouldin index and the optimal number of clusters
    """
    n_clus, db = np.loadtxt(scores_csv, unpack=True, delimiter=',', usecols=(0, 3))

    # Plot the Davies-Bouldin index and the optimal number of clusters
    all_indices = np.argsort(db)
    min_db_index = all_indices[0]
    min_db = n_clus[min_db_index]
    all_indices = np.delete(all_indices, 0)
    second_min_index = all_indices[0]
    second_min_db = n_clus[second_min_index]

    fig, ax = plt.subplots()
    ax.plot(n_clus, db, color='#005cde', label='DBI', linewidth=2)
    ax.set_xlabel('Cluster Number')
    ax.set_ylabel('Davies-Bouldin Index')
    ax.axvline(x=min_db, color='#de005c', linestyle='--', label=f'Optimal Cluster Number: {int(min_db)}', linewidth=2)
    ax.axvline(x=second_min_db, color='#00ab64', linestyle='--', label=f'Second Optimal Cluster Number: {int(second_min_db)}', linewidth=2)

    # Calculate the second derivative (before + after - 2*current)
    arr = db
    x = n_clus[1:-1]
    result = []
    for start_index, n_clusters in zip(range(1, len(arr) - 1), x):
        temp = arr[start_index + 1] + arr[start_index - 1] - (2 * arr[start_index])
        if arr[start_index] <= arr[start_index - 1] and arr[start_index] <= arr[start_index + 1]:
            result.append((n_clusters, temp))
    result = np.array(result)
    if len(result) == 0:
        print('No local minima found')
    elif len(result) >= 1:
        sorted_indices = np.argsort(result[:, 1])[::-1]
        sorted_result = result[sorted_indices]
        min_x = sorted_result[0][0]
        ax.axvline(x=min_x, color='#de8200', linestyle='--', label=f'Optimal 2nd deriv Cluster Number: {int(min_x)}', linewidth=2)
        if len(sorted_result) >= 2:
            sec_min_x = sorted_result[1][0]
            ax.axvline(x=sec_min_x, color='#6400ab', linestyle='--', label=f'Second Optimal 2nd deriv Cluster Number: {int(sec_min_x)}', linewidth=2)

    ax.legend(fontsize=10)
    plt.savefig(f'{output_dir}/{output_file_name}_summary.png', format='png', dpi=300)
    plt.show()

for scores_csv in scores_csvs:
    plot_scores(scores_csv)

In [ ]:
# @title ### **Cluster Assignment**
#@markdown This cell runs *k*-means with NANI initialization for the chosen number of clusters, assigns a cluster to every frame, and finds the frames closest to each cluster's medoid. It uses the same **sieve** as the screening cell.

import os

import numpy as np

from mdance.cluster.nani import KmeansNANI
from mdance.tools.bts import extended_comparison, calculate_medoid

#@markdown **n_clusters** is the number of clusters for labeling. Choose it from the plot above.
n_clusters = 18 #@param {type:"integer"}
#@markdown **n_structures** is the number of frames to extract from each cluster.
n_structures = 20 #@param {type:"integer"}

# K-means params
init_type = 'comp_sim'
metric = 'MSD'

os.makedirs(output_dir, exist_ok=True)
traj_numpy = np.load(output_name)[::sieve]
mod = KmeansNANI(data=traj_numpy, n_clusters=n_clusters, N_atoms=N_atoms, init_type=init_type,
                 metric=metric, percentage=10)
labels, centers, n_iter = mod.execute_kmeans_all()

# Renumber the clusters by size, largest first
sort_labels_by_size = np.argsort(np.bincount(labels))[::-1]
labels = np.array([np.where(sort_labels_by_size == i)[0][0] for i in labels])

# Find the frames closest to the medoid of each cluster
best_frames_indices = []
for label in np.unique(labels):
    cluster = np.where(labels == label)[0]
    if len(cluster) == 1:
        best = cluster
    else:
        medoid_index = calculate_medoid(traj_numpy[cluster], metric=metric, N_atoms=N_atoms)
        medoid = traj_numpy[cluster][medoid_index]
        msd_to_medoid = [extended_comparison(np.array([frame, medoid]), data_type='full', metric=metric,
                                             N_atoms=N_atoms) for frame in traj_numpy[cluster]]
        best = np.sort(cluster[np.argsort(msd_to_medoid)[:n_structures]])
    for frame in best:
        best_frames_indices.append((frame * sieve, label))
best_frames_indices = np.array(best_frames_indices)
np.savetxt(f'{output_dir}/best_frames_indices_{n_clusters}.csv', best_frames_indices, delimiter=',', fmt='%s',
           header=f'Number of clusters,{n_clusters}\nFrame Index,Cluster Index')

# Save cluster labels
with open(f'{output_dir}/labels_{n_clusters}.csv', 'w') as f:
    f.write(f'# init_type: {init_type}, Number of clusters: {n_clusters}\n')
    f.write('# Frame Index, Cluster Index\n')
    for i, row in enumerate(labels):
        f.write(f'{i * sieve},{row}\n')

# Calculate population of each cluster
with open(f'{output_dir}/summary_{n_clusters}.csv', 'w') as f:
    f.write(f'# Number of clusters, {n_clusters}\n')
    f.write('# Cluster Index, Fraction out of total frames\n')
    for i, row in enumerate(np.bincount(labels)):
        f.write(f'{i},{row/len(labels)}\n')

In [ ]:
# @title ### **Extract frames for each cluster**
#@markdown This cell writes a trajectory for each cluster containing its best frames (the frames closest to the cluster medoid).

import MDAnalysis as mda
import numpy as np

output_format = "dcd" #@param ["dcd", "nc", "xtc", "gro", "trr"]

u = mda.Universe(input_top, input_traj)
print(f'Number of atoms in the trajectory: {u.atoms.n_atoms}')

cluster_assignment = f'{output_dir}/best_frames_indices_{n_clusters}.csv'
x, y = np.loadtxt(cluster_assignment, delimiter=',', dtype=int, unpack=True)

for i in np.unique(y):
    # write trajectory with only the best frames of cluster i
    with mda.Writer(f'{output_dir}/best_frames_{i}.{output_format}', u.atoms.n_atoms) as W:
        for ts in u.trajectory[x[y == i]]:
            W.write(u.atoms)

In [ ]:
#@title **Download your results**
import os
import subprocess

from google.colab import files
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

if os.path.exists(f"results_MDANCE_{output_file_name}.zip"):
  os.remove(f"results_MDANCE_{output_file_name}.zip")
else:
  pass

subprocess.run(f"zip -FSr results_MDANCE_{output_file_name}.zip {output_dir}" , shell=True)

files.download(f"results_MDANCE_{output_file_name}.zip")

# **Which method should I use?**
This notebook used NANI. MDANCE includes other methods for different kinds of questions:

| Method | Use it when | Tutorial |
|---|---|---|
| NANI | You want *k*-means clustering with deterministic, well-spread initial centroids | [NANI tutorial](https://mdance.readthedocs.io/en/latest/tutorials/nani.html) |
| HELM | You want a hierarchy of clusters on a large dataset, built by merging many initial clusters | [HELM tutorial](https://mdance.readthedocs.io/en/latest/tutorials/helm.html) |
| eQual | You want flat clustering by a similarity threshold, without choosing the number of clusters | [eQual tutorial](https://mdance.readthedocs.io/en/latest/tutorials/equal.html) |
| DIVINE | You want top-down (divisive) clustering to look at the data at several resolutions | [DIVINE tutorial](https://github.com/mqcomplab/MDANCE/blob/main/tutorials/divine.md) |
| mdBIRCH | Your trajectory is very long or arrives as a stream | [mdBIRCH tutorial](https://github.com/mqcomplab/MDANCE/blob/main/tutorials/mdbirch.md) |
| SHINE | You want to cluster pathways, for example from enhanced sampling | [SHINE tutorial](https://github.com/mqcomplab/MDANCE/blob/main/tutorials/shine.md) |
| PRISM | You want to compare and cluster multiple pathways | [PRISM tutorial](https://github.com/mqcomplab/MDANCE/blob/main/tutorials/prism.md) |
| PRIME | You have clustering results and want to predict the native structure | [PRIME tutorial](https://mdance.readthedocs.io/en/latest/tutorials/prime.html) |